# Cleaned ChEMBL Data Preprocessing with EDA & Normalization

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install chembl_webresource_client
import pandas as pd
from chembl_webresource_client.new_client import new_client

target = new_client.target
target_query = target.search('coronavirus')
targets = pd.DataFrame.from_dict(target_query)
targets.head(5)

selected_protein_targets = targets.target_chembl_id[4]
selected_protein_targets

# Retrieve Bioactivity data for the selected targets
bioactivity_data = new_client.activity
# Filter data for those specific targets and set standard measuement unit to IC50 measurements
filtered_data = bioactivity_data.filter(target_chembl_id=selected_protein_targets).filter(standat_type="IC50")
# Create DataFrame from the filtered data stored in a dictionary, remove None/NA and then store it in a csv file for reusability
bioactivity_DF = pd.DataFrame.from_dict(filtered_data)
bioactivity_DF = bioactivity_DF[bioactivity_DF.standard_value.notna()]
bioactivity_DF.to_csv('/content/drive/MyDrive/Colab Notebooks/DTI/data/raw_bioactivity_data.csv', index = False)
bioactivity_DF.head(5)

activity_classes = []
for i in bioactivity_DF.standard_value:
  if float(i) >= 10000:
    activity_classes.append("inactive")
  elif float(i) <= 1000:
    activity_classes.append("active")
  else:
    activity_classes.append("moderate")

## Select columns of interest
columns = ['molecule_chembl_id','canonical_smiles', 'standard_value']
bioactivity_DF = bioactivity_DF[columns]
bioactivity_DF['activity_class'] = activity_classes
bioactivity_DF.head(10)

## Saving preprocessed data to csv
bioactivity_DF.to_csv('preprocessed_bioactivity_data.csv', index= False)

In [4]:
  # Performing Exploratory Data Analysis (EDA)
  print("Dataset Information:")
  print(bioactivity_DF.info())

  print("Statistical Summary:")
  print(bioactivity_DF.describe())

  # Normalization using Min-Max Scaling
  from sklearn.preprocessing import MinMaxScaler
  scaler = MinMaxScaler()
  bioactivity_DF['standard_value_normalized'] = scaler.fit_transform(bioactivity_DF[['standard_value']])

  # Saving cleaned and processed data
  bioactivity_DF.to_csv('cleaned_bioactivity_data.csv', index=False)

  # Display first few rows
  bioactivity_DF.head()


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, 0 to 257
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   molecule_chembl_id  201 non-null    object
 1   canonical_smiles    201 non-null    object
 2   standard_value      201 non-null    object
 3   activity_class      201 non-null    object
dtypes: object(4)
memory usage: 7.9+ KB
None
Statistical Summary:
       molecule_chembl_id                                   canonical_smiles  \
count                 201                                                201   
unique                150                                                150   
top          CHEMBL217092  CN1CCN(CCOc2cc(OC3CCOCC3)c3c(Nc4c(Cl)ccc5c4OCO...   
freq                    6                                                  6   

       standard_value activity_class  
count             201            201  
unique             96              3  
top     

,molecule_chembl_id,canonical_smiles,standard_value,activity_class,standard_value_normalized
0,CHEMBL20636,CCOC(=O)/C=C/[C@H](C[C@@H]1CCNC1=O)NC(=O)[C@H]...,10000.0,inactive,0.089274
1,CHEMBL213054,CC(OC(C)(C)C)[C@H](NC(=O)OCc1ccccc1)C(=O)N[C@@...,68.0,active,0.000594
2,CHEMBL2441745,CC(C)C[C@H](NC(=O)[C@H](Cc1cccc2ccccc12)NC(=O)...,500.0,active,0.004451
3,CHEMBL2441741,CC(C)C[C@H](NC(=O)[C@H](Cc1cccc2ccccc12)NC(=O)...,200.0,active,0.001773
4,CHEMBL1643,NC(=O)c1ncn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)n1,112000.0,inactive,1.000000
